# Technical-Regulatory RAG for FM Spectrum Monitoring

This tutorial implements a complete, hardware-agnostic RAG workflow over a normalized FM monitoring framework and a small authoritative ANE/FCC teaching subset. It performs structural chunking, authority-aware retrieval, objective evaluation, and grounded local generation without physical SDR hardware, private API keys, LangChain, or runtime web scraping.

**Canonical editable inputs:** three Markdown files under `RAG-research/corpus/`.  
**Portable input:** a hash-verified compressed copy of those same files embedded in this notebook.  
**Outputs:** 40 indexed chunks, retrieval metrics, grounded answers, deterministic evidence metadata, abstention and adversarial-policy results.  
**Execution validation:** Validated end-to-end in a clean Google Colab CPU runtime using the embedded standalone corpus.

## 1. Problem Definition

The knowledge domain is an SDR-based FM spectrum-monitoring framework: calibration, measurands, DSP responsibilities, uncertainty, decision rules, evidence, and distributed monitoring. This is **not** an SDR hardware tutorial. No physical receiver is required, and the implementation remains independent of receiver model, processing platform, or fixed acquisition configuration.

Plain semantic search is insufficient for regulatory knowledge because similar language can carry different authority. A framework may explain *how* to monitor, a regulation may establish a binding jurisdictional limit, a technical standard may supply international engineering context, and a metrology source may define uncertainty practice. Retrieval must preserve those roles and must not let a framework answer as though it were a regulator.

Jurisdiction is therefore part of the meaning of a result. A numerical limit is usable only when retrieved from the applicable authoritative regulatory source and linked to its jurisdiction, provision, and effective version.

## 2. Learning Objectives

Readers will learn to normalize and structurally chunk technical-regulatory documents, preserve authority metadata, embed and index them, compare semantic and policy-aware retrieval, calculate retrieval metrics, and generate evidence-grounded answers. The notebook also demonstrates deterministic abstention and why a generator must never decide jurisdictional authority.

## 3. Runtime and Environment

One setup cell detects Colab without importing it as a dependency, installs only missing runtime packages through the active notebook kernel, imports the resolved libraries, and reports versions. Repository and standalone modes use the same downstream pipeline.

### Environment setup

**Intent:** detect runtime, conditionally install the six declared packages, and import resolved dependencies.  
**Inputs:** active Python/Jupyter environment.  
**Expected output:** execution mode, Python/working-directory information, and package versions.  
**Side effects:** missing packages are installed into the active notebook kernel. This is the only environment/setup cell.

In [ ]:
from pathlib import Path
from collections import Counter
import base64
import gzip
import hashlib
import importlib
import importlib.util
import json
import platform
import re

IN_COLAB = importlib.util.find_spec("google.colab") is not None
EXECUTION_MODE = "COLAB" if IN_COLAB else "LOCAL"
REQUIRED_DISTRIBUTIONS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sentence-transformers": "sentence_transformers",
    "chromadb": "chromadb",
    "transformers": "transformers",
    "sentencepiece": "sentencepiece",
}
missing_distributions = [
    distribution for distribution, module in REQUIRED_DISTRIBUTIONS.items()
    if importlib.util.find_spec(module) is None
]
if missing_distributions:
    try:
        active_ipython = get_ipython()
    except NameError as exc:
        raise RuntimeError(
            "Install missing runtime packages: " + ", ".join(missing_distributions)
        ) from exc
    active_ipython.run_line_magic(
        "pip", "install --quiet " + " ".join(missing_distributions)
    )

import numpy as np
import pandas as pd
import sentence_transformers
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import transformers
import sentencepiece

print(f"Execution mode: {EXECUTION_MODE}")
print(f"Python version: {platform.python_version()}")
print(f"Working directory: {Path.cwd().resolve()}")
for name, version in {
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sentence_transformers": sentence_transformers.__version__,
    "chromadb": chromadb.__version__,
    "transformers": transformers.__version__,
    "sentencepiece": sentencepiece.__version__,
}.items():
    print(f"{name}: {version}")


## 4. Corpus and Source Hierarchy

TASK-04 indexes only normalized local sources:

- `../corpus/framework/03SpectrumSensingFM0_RAG_ready.md`
- `../corpus/regulation/CO/ANE_PTNRS_FM_current.md`
- `../corpus/regulation/US/FCC_Part73_FM_selected.md`

The regulatory files are narrow teaching subsets built from official current web sources and stored locally so normal execution never depends on runtime scraping. URLs are provenance only. The notebook is not legal advice, and the subset does not replace current compilations, official editions, station authorizations, amendments, or other applicable provisions.

| Role | Purpose | Numeric-limit authority |
|---|---|---|
| `framework` | Monitoring architecture, measurands, workflow, and evidence model | No |
| `regulation` | Binding jurisdiction-specific requirements when issued by the governing authority | Yes, within the retained rule's exact applicability |
| `technical_standard` | International or industry technical context | Only when legally adopted and correctly scoped |
| `metrology` | Calibration, traceability, uncertainty, and decision-rule methods | No jurisdictional FM limit by itself |

TASK-04 adds no metrology, ITU, ISO, generation, or external runtime ingestion.

The embedded bundle exists only for single-file teaching portability. Repository Markdown files remain the canonical editable sources; rebuilding the notebook must regenerate the bundle and verify hashes.

### Corpus bootstrap contract

**Intent:** define canonical relative paths, the embedded gzip/base64 bundle, and expected SHA-256 identities.  
**Inputs:** notebook constant created from the canonical Markdown files.  
**Expected output:** reusable repository discovery and embedded decoding functions.  
**Side effects:** none; standalone reconstruction is in memory.

In [ ]:
EXECUTION_ATTEMPT_ID = "TASK05_FRESH_PYTHON_001"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GENERATOR_MODEL_NAME = "google/flan-t5-small"

CORPUS_RELATIVE_PATHS = {
    "framework": Path("RAG-research/corpus/framework/03SpectrumSensingFM0_RAG_ready.md"),
    "CO": Path("RAG-research/corpus/regulation/CO/ANE_PTNRS_FM_current.md"),
    "US": Path("RAG-research/corpus/regulation/US/FCC_Part73_FM_selected.md"),
}
EXPECTED_CORPUS_SHA256 = {
    "framework": "b1849206cce0a8f3e6f3b04df4c70b5d922b3df10437071433a7d2beafa949be",
    "CO": "b1dc7047ab49ffc4461e8bfc00a4c5014e5c5db813ceda31d3a81f7a99f5d37d",
    "US": "ed936de528f2f8076bca9d10ca033432287684cc84c1c4bf5f38b28605da6c6d"
}
EMBEDDED_CORPUS_B64 = "..."

FRAMEWORK_DEFAULTS = {
    "source_id": "03SpectrumSensingFM0_RAG_READY",
    "source_role": "framework",
    "jurisdiction": "NONE",
    "authority": "03SpectrumSensingFM0 framework",
    "regulatory_section": "NONE",
    "source_url": "LOCAL_NORMALIZED_FRAMEWORK",
    "source_version": "RAG_READY",
    "status": "framework",
    "numeric_limit_authority": False,
    "contains_numeric_limit": False,
}

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def decode_embedded_corpus() -> dict[str, str]:
    compressed = base64.b64decode(EMBEDDED_CORPUS_B64.encode("ascii"))
    payload = gzip.decompress(compressed).decode("utf-8")
    decoded = json.loads(payload)
    assert set(decoded) == set(CORPUS_RELATIVE_PATHS)
    assert {
        name: sha256_text(text) for name, text in decoded.items()
    } == EXPECTED_CORPUS_SHA256
    return decoded

def find_repository_root(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if all((candidate / path).is_file() for path in CORPUS_RELATIVE_PATHS.values()):
            return candidate
        if candidate.name == "RAG-research":
            candidate = candidate.parent
            if all((candidate / path).is_file() for path in CORPUS_RELATIVE_PATHS.values()):
                return candidate
    return None

def load_corpus(force_embedded: bool = False) -> tuple[dict[str, str], str]:
    embedded = decode_embedded_corpus()
    root = None if force_embedded else find_repository_root(Path.cwd())
    if root is None:
        return embedded, "EMBEDDED_STANDALONE"
    repository_texts = {
        name: (root / relative).read_text(encoding="utf-8")
        for name, relative in CORPUS_RELATIVE_PATHS.items()
    }
    repository_hashes = {
        name: sha256_text(text) for name, text in repository_texts.items()
    }
    assert repository_hashes == EXPECTED_CORPUS_SHA256
    assert repository_hashes == {
        name: sha256_text(text) for name, text in embedded.items()
    }
    return repository_texts, "REPOSITORY"


## 5. Document Normalization

**Intent:** load canonical files when present and otherwise use the embedded byte-equivalent corpus; independently exercise standalone decoding.  
**Inputs:** bootstrap functions and hashes.  
**Expected output:** corpus mode, three normalized texts, and repository/embedded hash evidence.  
**Side effects:** none. No network, clone, token, Drive, or manual corpus upload is required.

In [ ]:
corpus_texts, CORPUS_MODE = load_corpus()
standalone_corpus_texts, standalone_test_mode = load_corpus(force_embedded=True)
assert standalone_test_mode == "EMBEDDED_STANDALONE"
assert set(standalone_corpus_texts) == {"framework", "CO", "US"}
assert {
    name: sha256_text(text) for name, text in standalone_corpus_texts.items()
} == EXPECTED_CORPUS_SHA256

source_text = corpus_texts["framework"]
print(f"Corpus mode: {CORPUS_MODE}")
for source_name, text in corpus_texts.items():
    print(f"{source_name}: {len(text):,} characters; SHA-256 {sha256_text(text)}")
print("Standalone embedded bootstrap: PASS (3/3 sources)")


### Validate source structure and domain coverage

**Intent:** verify the expected framework sections, all seven measurands, and hardware-agnostic active content before chunking.  
**Inputs:** `source_text` and explicit expected labels.  
**Expected output:** passed source checks and no prohibited-token matches outside Appendix C.  
**Side effects:** none. Failing before chunk creation prevents malformed or out-of-scope knowledge from entering retrieval.

In [ ]:
REQUIRED_H2_HEADINGS = [
    "1. Document Identity and Provenance",
    "2. Purpose and Scope",
    "3. Regulatory and Standards Context",
    "4. System-Level Requirements",
    "5. Calibration and Traceability Hierarchy",
    "6. Hardware-Agnostic Functional Architecture",
    "7. FM Compliance Measurement Model",
    "8. Measurement Capability Classification",
    "9. Measurement Uncertainty Requirements",
    "10. Node-Level DSP Knowledge Pipeline",
    "11. Regulatory Rule Evaluation",
    "12. Reporting, Evidence and Provenance",
    "13. Distributed / Multi-Node Monitoring",
    "Appendix A — Measurement Uncertainty",
    "Appendix B — Generic Decision Rules",
    "Appendix C — Transformation Log",
]

PRIMARY_MEASURANDS = [
    "carrier frequency error",
    "calibrated received power",
    "field strength",
    "occupied bandwidth",
    "adjacent-channel / out-of-band emission level",
    "peak deviation / multiplex-related indicator",
    "channel occupancy over time",
]

missing_headings = [
    heading for heading in REQUIRED_H2_HEADINGS
    if f"## {heading}" not in source_text
]
missing_measurands = [
    measurand for measurand in PRIMARY_MEASURANDS
    if measurand.casefold() not in source_text.casefold()
]

active_source_text = source_text.split("## Appendix C — Transformation Log", maxsplit=1)[0]
PROHIBITED_DEVICE_PATTERN = re.compile(
    r"\b(?:HackRF|Dragonboard|Raspberry(?:\s+Pi)?|RPi|GNU\s+Radio|Airspy|USRP|RTL-SDR|USB)\b|20\s*MS/s",
    flags=re.IGNORECASE,
)
active_hardware_matches = sorted(set(PROHIBITED_DEVICE_PATTERN.findall(active_source_text)))

assert not missing_headings, f"Missing required sections: {missing_headings}"
assert not missing_measurands, f"Missing primary measurands: {missing_measurands}"
assert not active_hardware_matches, (
    "Hardware-specific tokens found outside the Transformation Log: "
    f"{active_hardware_matches}"
)

print(f"Required sections present: {len(REQUIRED_H2_HEADINGS)}/{len(REQUIRED_H2_HEADINGS)}")
print(f"Primary measurands present: {len(PRIMARY_MEASURANDS)}/{len(PRIMARY_MEASURANDS)}")
print("Hardware-agnostic active-source check: PASSED")


## 6. Semantic and Structural Chunking

The main strategy follows Markdown headings and named conceptual boundaries; it does not cut text at arbitrary character counts. Calibration, measurands, uncertainty equations, and decision rules remain attached to their assumptions and applicability explanations.

**Implementation decision:** this iteration uses an explicit extraction manifest. It is deliberately inspectable: each chunk has a stable ID, source-page mapping, semantic topic, and source boundary. If the normalized framework structure changes, extraction fails loudly instead of silently returning misleading fragments.

Data partitioning and a split manifest are not applicable: there is one authoritative input document and no training/evaluation split. Model definition and training are also not applicable in TASK-02.

### Parse Markdown headings

**Intent:** build independent heading indexes for the framework and both curated regulatory sources.  
**Inputs:** normalized texts in `corpus_texts`.  
**Expected output:** deterministic maps from heading titles to complete section text.  
**Side effects:** none. Rule-level structural indexing keeps numeric values attached to their section and applicability metadata.

In [ ]:
HEADING_PATTERN = re.compile(r"^(#{1,6})\s+(.+?)\s*$", flags=re.MULTILINE)

def build_heading_index(markdown_text: str) -> dict[str, str]:
    matches = list(HEADING_PATTERN.finditer(markdown_text))
    sections = {}
    for index, match in enumerate(matches):
        level = len(match.group(1))
        title = match.group(2)
        end = len(markdown_text)
        for following in matches[index + 1:]:
            if len(following.group(1)) <= level:
                end = following.start()
                break
        sections[title] = markdown_text[match.start():end].strip()
    return sections

heading_indexes = {
    source_name: build_heading_index(text)
    for source_name, text in corpus_texts.items()
}
heading_sections = heading_indexes["framework"]

def require_section(title: str, source_name: str = "framework") -> str:
    try:
        text = heading_indexes[source_name][title].strip()
    except KeyError as exc:
        raise KeyError(f"Required section not found in {source_name}: {title}") from exc
    if not text:
        raise ValueError(f"Required section is empty in {source_name}: {title}")
    return text

def require_prefixed_section(prefix: str, source_name: str) -> str:
    matches = [
        text
        for title, text in heading_indexes[source_name].items()
        if title.startswith(prefix)
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one {source_name} heading beginning {prefix!r}; found {len(matches)}"
        )
    return matches[0].strip()

for source_name, sections in heading_indexes.items():
    print(f"{source_name} indexed headings: {len(sections)}")


### Define conceptual-boundary helpers

**Intent:** extract named concepts inside larger sections without separating definitions from their explanatory paragraphs.  
**Inputs:** complete structural section text plus exact start/end markers.  
**Expected output:** non-empty semantic fragments or an immediate exception.  
**Side effects:** none. Explicit boundaries reduce accidental topic mixing in later retrieval.

In [ ]:
def extract_between(text: str, start_marker: str, end_marker: str | None = None) -> str:
    start = text.find(start_marker)
    if start < 0:
        raise ValueError(f"Start marker not found: {start_marker!r}")
    end = len(text) if end_marker is None else text.find(end_marker, start + len(start_marker))
    if end_marker is not None and end < 0:
        raise ValueError(f"End marker not found: {end_marker!r}")
    fragment = text[start:end].strip()
    if not fragment:
        raise ValueError(f"Empty fragment for marker: {start_marker!r}")
    return fragment

dsp_section = require_section("10. Node-Level DSP Knowledge Pipeline")
array_section = require_section("13. Distributed / Multi-Node Monitoring")

dsp_boundaries = {
    "FW_DSP_ACQUISITION": (
        "1. **Wideband Acquisition and Conditioning**",
        "2. **Channelization, Isolation and Tracking**",
    ),
    "FW_DSP_CHANNELIZATION": (
        "2. **Channelization, Isolation and Tracking**",
        "3. **Optional Per-Channel Demodulation and Identification**",
    ),
    "FW_DSP_DEMOD_IDENT": (
        "3. **Optional Per-Channel Demodulation and Identification**",
        "4. **Compliance Measurand Estimation**",
    ),
    "FW_DSP_MEAS_ESTIMATION": (
        "4. **Compliance Measurand Estimation**",
        "5. **Rule Evaluation and Alerting**",
    ),
    "FW_DSP_RULE_EVALUATION": (
        "5. **Rule Evaluation and Alerting**",
        "6. **Reporting and Evidence Archiving**",
    ),
    "FW_DSP_REPORTING": (
        "6. **Reporting and Evidence Archiving**",
        None,
    ),
}

array_boundaries = {
    "inter_node": ("**Inter-node calibration.**", "**Timing alignment.**"),
    "timing": ("**Timing alignment.**", "**Frequency-reference coordination.**"),
    "reference": ("**Frequency-reference coordination.**", "**Health coordination.**"),
    "health": ("**Health coordination.**", "**Cross-node consistency.**"),
    "consistency": ("**Cross-node consistency.**", "**Fusion rules.**"),
    "fusion": ("**Fusion rules.**", "**Array-level traceability.**"),
    "evidence": ("**Array-level traceability.**", "**Degraded operation.**"),
    "degraded": ("**Degraded operation.**", None),
}

print("Conceptual boundary contracts: PASSED")


### Assemble deterministic framework chunks and extended metadata

**Intent:** preserve the validated 31-chunk framework corpus while adding stable sentinel fields required by the combined index.  
**Inputs:** framework structural sections and conceptual fragments.  
**Expected output:** exactly 31 `framework_chunks`, including seven measurand chunks.  
**Side effects:** none. Sentinels make the metadata schema uniform without granting the framework regulatory authority.

In [ ]:
def make_chunk(
    chunk_id: str,
    text: str,
    section: str,
    topic: str,
    source_pages: str,
    measurand: str | None = None,
    **metadata_overrides,
) -> dict:
    chunk = {
        "chunk_id": chunk_id,
        "text": text.strip(),
        "section": section,
        "topic": topic,
        "measurand": measurand,
        "source_pages": source_pages,
        **FRAMEWORK_DEFAULTS,
    }
    chunk.update(metadata_overrides)
    return chunk

framework_chunk_specs = [
    ("FW_CAL_TIER1", "5.1 Tier 1 — Primary Laboratory Calibration", "Tier 1 primary laboratory calibration", "2–3", None),
    ("FW_CAL_TIER2", "5.2 Tier 2 — Secondary Field Verification", "Tier 2 secondary field verification", "2–3", None),
    ("FW_CAL_TIER3", "5.3 Tier 3 — Relative Channel Calibration", "Tier 3 relative channel calibration", "2–3", None),
    ("FW_MEAS_CARRIER_FREQ", "7.3 Carrier Frequency Error", "carrier frequency error", "5–6", "carrier frequency error"),
    ("FW_MEAS_RX_POWER", "7.4 Calibrated Received Power", "calibrated received power", "5–6", "calibrated received power"),
    ("FW_MEAS_FIELD_STRENGTH", "7.5 Field Strength", "field strength", "5–6", "field strength"),
    ("FW_MEAS_OCC_BANDWIDTH", "7.6 Occupied Bandwidth", "occupied bandwidth", "5–6", "occupied bandwidth"),
    ("FW_MEAS_ADJ_OOB", "7.7 Adjacent-Channel / Out-of-Band Emission Level", "adjacent-channel / out-of-band emission level", "5–6", "adjacent-channel / out-of-band emission level"),
    ("FW_MEAS_PEAK_MPX", "7.8 Peak Deviation / Multiplex-Related Indicator", "peak deviation / multiplex-related indicator", "5–6", "peak deviation / multiplex-related indicator"),
    ("FW_MEAS_OCCUPANCY", "7.9 Channel Occupancy Over Time", "channel occupancy over time", "5–6", "channel occupancy over time"),
    ("FW_CAPABILITY_CLASS", "8. Measurement Capability Classification", "measurement capability classification", "7", None),
    ("FW_UNC_TYPE_A", "A.1 Type A uncertainty", "Type A uncertainty", "29", None),
    ("FW_UNC_TYPE_B", "A.2 Type B uncertainty", "Type B uncertainty", "29", None),
    ("FW_UNC_COMBINED", "A.3 Combined standard uncertainty and correlations", "combined uncertainty and covariance", "29–30", None),
    ("FW_UNC_EXPANDED", "A.4 Expanded uncertainty", "expanded uncertainty", "30", None),
    ("FW_UNC_REEVALUATION", "A.5 Periodic re-evaluation", "periodic uncertainty re-evaluation", "30", None),
    ("FW_DEC_SIMPLE", "B.1 Simple threshold decision rule", "simple threshold decision rule", "30–31", None),
    ("FW_DEC_GUARD_BAND", "B.2 Conservative guard-band decision rule", "conservative guard-band decision rule", "30–31", None),
    ("FW_DEC_SHARED_RISK", "B.3 Shared-risk decision rule", "shared-risk decision rule", "31–32", None),
]

framework_chunks = [
    make_chunk(chunk_id, require_section(section), section, topic, pages, measurand)
    for chunk_id, section, topic, pages, measurand in framework_chunk_specs
]

framework_chunks.append(make_chunk(
    "FW_OBS_PRIMARY_SECONDARY",
    require_section("7. FM Compliance Measurement Model")
    + "\n\n"
    + require_section("7.2 Secondary Observables"),
    "7 / 7.2 Primary and Secondary Observables",
    "primary compliance measurands versus secondary observables",
    "5",
))

dsp_topics = {
    "FW_DSP_ACQUISITION": "wideband acquisition and conditioning",
    "FW_DSP_CHANNELIZATION": "channelization, isolation and tracking",
    "FW_DSP_DEMOD_IDENT": "optional per-channel demodulation and identification",
    "FW_DSP_MEAS_ESTIMATION": "compliance measurand estimation",
    "FW_DSP_RULE_EVALUATION": "rule evaluation and alerting",
    "FW_DSP_REPORTING": "reporting and evidence archiving",
}
for chunk_id, (start, end) in dsp_boundaries.items():
    framework_chunks.append(make_chunk(
        chunk_id,
        extract_between(dsp_section, start, end),
        "10. Node-Level DSP Knowledge Pipeline",
        dsp_topics[chunk_id],
        "8–10",
    ))

array_fragments = {
    name: extract_between(array_section, start, end)
    for name, (start, end) in array_boundaries.items()
}
framework_chunks.extend([
    make_chunk("FW_ARRAY_INTER_NODE_CAL", array_fragments["inter_node"], "13. Distributed / Multi-Node Monitoring", "inter-node calibration", "11–12"),
    make_chunk("FW_ARRAY_TIMING_REFERENCE", array_fragments["timing"] + "\n\n" + array_fragments["reference"], "13. Distributed / Multi-Node Monitoring", "timing and frequency-reference coordination", "11–12"),
    make_chunk("FW_ARRAY_HEALTH", array_fragments["health"], "13. Distributed / Multi-Node Monitoring", "distributed node health coordination", "11–12"),
    make_chunk("FW_ARRAY_CONSISTENCY_FUSION", array_fragments["consistency"] + "\n\n" + array_fragments["fusion"], "13. Distributed / Multi-Node Monitoring", "cross-node consistency and fusion", "11–12"),
    make_chunk("FW_ARRAY_EVIDENCE_DEGRADED", array_fragments["evidence"] + "\n\n" + array_fragments["degraded"], "13. Distributed / Multi-Node Monitoring", "array evidence and degraded operation", "11–12"),
])

chunks = framework_chunks
print(f"Framework structural chunks: {len(framework_chunks)}")


## 7. Metadata Schema

Every chunk retains the TASK-03 fields and adds regulatory provenance:

```python
{
    "chunk_id": str,
    "text": str,
    "source_id": str,
    "source_role": str,
    "section": str,
    "topic": str,
    "measurand": str | None,
    "jurisdiction": str,
    "authority": str,
    "source_pages": str,
    "numeric_limit_authority": bool,
    "regulatory_section": str,
    "source_url": str,
    "source_version": str,
    "status": str,
    "contains_numeric_limit": bool,
}
```

`numeric_limit_authority` means the source is competent to supply a regulatory limit within its scope; `contains_numeric_limit` states whether that particular chunk actually retains one. Framework chunks use explicit sentinel values and never receive numeric-limit authority.

### Inspect the chunk catalog

**Intent:** render compact retrieval metadata without requiring pandas.  
**Inputs:** `chunks`.  
**Expected output:** a Markdown-style table containing chunk ID, topic, measurand, role, jurisdiction, and text length.  
**Side effects:** none. Catalog inspection makes coverage gaps, oversized concepts, and authority mistakes visible before embedding.

In [ ]:
catalog_columns = ["chunk_id", "topic", "measurand", "source_role", "jurisdiction", "text_length"]
catalog_rows = [
    {
        "chunk_id": chunk["chunk_id"],
        "topic": chunk["topic"],
        "measurand": chunk["measurand"] or "—",
        "source_role": chunk["source_role"],
        "jurisdiction": chunk["jurisdiction"],
        "text_length": len(chunk["text"]),
    }
    for chunk in chunks
]

def print_markdown_table(rows: list[dict], columns: list[str]) -> None:
    widths = {
        column: max(len(column), *(len(str(row[column])) for row in rows))
        for column in columns
    }
    print("| " + " | ".join(column.ljust(widths[column]) for column in columns) + " |")
    print("|-" + "-|-".join("-" * widths[column] for column in columns) + "-|")
    for row in rows:
        print("| " + " | ".join(str(row[column]).ljust(widths[column]) for column in columns) + " |")

print_markdown_table(catalog_rows, catalog_columns)


### Display representative chunks

**Intent:** inspect complete text for one measurand, one uncertainty concept, and one DSP stage.  
**Inputs:** deterministic chunk IDs.  
**Expected output:** three full chunks with metadata and untruncated text.  
**Side effects:** none. Full inspection verifies semantic coherence and confirms that formulas remain with their assumptions.

In [ ]:
chunks_by_id = {chunk["chunk_id"]: chunk for chunk in chunks}
representative_ids = [
    "FW_MEAS_FIELD_STRENGTH",
    "FW_UNC_COMBINED",
    "FW_DSP_CHANNELIZATION",
]

for chunk_id in representative_ids:
    chunk = chunks_by_id[chunk_id]
    print("=" * 88)
    print(f"{chunk_id} | {chunk['topic']} | pages {chunk['source_pages']}")
    print("-" * 88)
    print(chunk["text"])
    print()


### Execute framework corpus and metadata assertions

**Intent:** preserve every TASK-02/TASK-03 framework invariant before regulatory content is added.  
**Inputs:** `framework_chunks`, expected measurands, and prohibited-token pattern.  
**Expected output:** a concise framework validation summary.  
**Side effects:** none. This gate prevents regulatory extension from silently altering the established framework corpus.

In [ ]:
measurand_chunks = [
    chunk for chunk in framework_chunks
    if chunk["measurand"] is not None
]
framework_chunk_ids = [chunk["chunk_id"] for chunk in framework_chunks]
decision_chunk_ids = {"FW_DEC_SIMPLE", "FW_DEC_GUARD_BAND", "FW_DEC_SHARED_RISK"}

assert len(framework_chunks) == 31
assert len(measurand_chunks) == 7
assert {chunk["measurand"] for chunk in measurand_chunks} == set(PRIMARY_MEASURANDS)
assert all(chunk_id.strip() for chunk_id in framework_chunk_ids)
assert len(framework_chunk_ids) == len(set(framework_chunk_ids))
assert all(chunk["source_role"] == "framework" for chunk in framework_chunks)
assert all(chunk["jurisdiction"] == "NONE" for chunk in framework_chunks)
assert all(chunk["numeric_limit_authority"] is False for chunk in framework_chunks)
assert all(chunk["contains_numeric_limit"] is False for chunk in framework_chunks)
assert all(chunk["text"].strip() for chunk in framework_chunks)
assert not [
    chunk["chunk_id"]
    for chunk in framework_chunks
    if PROHIBITED_DEVICE_PATTERN.search(chunk["text"])
]
assert any(chunk["chunk_id"].startswith("FW_UNC_") for chunk in framework_chunks)
assert decision_chunk_ids.issubset(set(framework_chunk_ids))

validation_summary = {
    "source_structure": "PASSED",
    "framework_chunks": len(framework_chunks),
    "primary_measurand_chunks": len(measurand_chunks),
    "framework_authority_constraints": "PASSED",
    "active_chunk_hardware_agnostic": "PASSED",
    "overall": "PASSED",
}
print("Framework validation summary")
for check, result in validation_summary.items():
    print(f"- {check}: {result}")


### Short exercise: inspect authority before retrieval

Choose any chunk and answer: can it provide a jurisdiction-specific numerical FM limit? Use `source_role`, `jurisdiction`, and `numeric_limit_authority` rather than guessing from the prose.

**Answer scaffold:** For every current chunk the answer is “no”: it is a framework source, has no jurisdiction, and explicitly lacks numeric-limit authority. A later regulation chunk would require authoritative provenance before that answer could change.

### Create curated regulatory chunks

Regulatory chunking follows rule headings rather than character counts. Each chunk retains its authority, jurisdiction, source URL, version, section, applicability statement, and any numeric value as one coherent unit.

**Intent:** convert the five ANE and four verified FCC sections into deterministic regulatory chunks.  
**Inputs:** the two normalized local regulatory Markdown sources.  
**Expected output:** non-empty `co_regulatory_chunks` and `us_regulatory_chunks` with complete provenance.  
**Side effects:** none. Keeping applicability beside each value prevents a limit from being detached from its legal scope.

In [ ]:
CO_SOURCE_URL = "https://normograma.mintic.gov.co/mintic/compilacion/docs/resolucion_ane_0105_2020.htm"
US_URLS = {
    "73.1545": "https://www.ecfr.gov/current/title-47/chapter-I/subchapter-C/part-73/subpart-H/section-73.1545",
    "73.314": "https://www.ecfr.gov/current/title-47/chapter-I/subchapter-C/part-73/subpart-B/section-73.314",
    "73.1570": "https://www.ecfr.gov/current/title-47/chapter-I/subchapter-C/part-73/subpart-H/section-73.1570",
}

co_specs = [
    ("CO_ANE_SCOPE_FM_BAND", "Article 2.2.1 / Annex 2 scope", "FM regulatory scope and band", None, "Article 2.2.1; Annex 2 scope", True),
    ("CO_ANE_FREQ_OPERATION", "Section 5.1.1 Frecuencia de Operación", "carrier frequency tolerance", "carrier frequency error", "Annex 2 §5.1.1", True),
    ("CO_ANE_MAX_DEVIATION", "Section 5.1.2 Excursión Máxima de Frecuencia", "maximum FM frequency deviation", "peak deviation / multiplex-related indicator", "Annex 2 §5.1.2", True),
    ("CO_ANE_OOB_EMISSIONS", "Section 5.1.4 Unwanted Emissions", "out-of-band and spurious emissions", "adjacent-channel / out-of-band emission level", "Annex 2 §§5.1.4–5.1.4.2", True),
    ("CO_ANE_NECESSARY_BW", "Section 5.1.5 Necessary and Occupied Bandwidth", "necessary and occupied bandwidth", "occupied bandwidth", "Annex 2 §5.1.5", True),
]
co_regulatory_chunks = [
    make_chunk(
        chunk_id,
        require_prefixed_section(chunk_id, "CO"),
        section_title,
        topic,
        "ONLINE",
        measurand,
        source_id="ANE_PTNRS_FM_CURRENT",
        source_role="regulation",
        jurisdiction="CO",
        authority="ANE",
        regulatory_section=regulatory_section,
        source_url=CO_SOURCE_URL,
        source_version="MinTIC current compilation accessed 2026-08-14; Resolution 406/2026 amendment context",
        status="current_compilation",
        numeric_limit_authority=True,
        contains_numeric_limit=contains_limit,
    )
    for chunk_id, section_title, topic, measurand, regulatory_section, contains_limit in co_specs
]

us_specs = [
    ("US_FCC_FREQ_TOLERANCE", "§73.1545(b) FM carrier-frequency departure", "carrier frequency tolerance", "carrier frequency error", "47 CFR §73.1545(b)(1)–(2)", US_URLS["73.1545"], True),
    ("US_FCC_FIELD_STRENGTH_CAL", "§73.314 field-strength procedure and calibration check", "field strength calibration procedure", "field strength", "47 CFR §73.314(a), (b)(2), (c)(2)", US_URLS["73.314"], False),
    ("US_FCC_FIELD_STRENGTH_REPORT", "§73.314 calibration and equipment reporting", "field strength calibration reporting", "field strength", "47 CFR §73.314(b)(3)(iv)–(v), (c)(3)(vi)–(vii)", US_URLS["73.314"], False),
    ("US_FCC_MODULATION", "§73.1570(b)(2) FM modulation levels", "FM modulation and peak deviation limits", "peak deviation / multiplex-related indicator", "47 CFR §73.1570(b)(2)(i)–(ii)", US_URLS["73.1570"], True),
]
us_regulatory_chunks = [
    make_chunk(
        chunk_id,
        require_prefixed_section(chunk_id, "US"),
        section_title,
        topic,
        "ONLINE",
        measurand,
        source_id="FCC_PART73_FM_SELECTED",
        source_role="regulation",
        jurisdiction="US",
        authority="FCC",
        regulatory_section=regulatory_section,
        source_url=source_url,
        source_version="eCFR Title 47 current through 2026-08-12; accessed 2026-08-14",
        status="current",
        numeric_limit_authority=True,
        contains_numeric_limit=contains_limit,
    )
    for chunk_id, section_title, topic, measurand, regulatory_section, source_url, contains_limit in us_specs
]

regulatory_chunks = co_regulatory_chunks + us_regulatory_chunks
chunks = framework_chunks + regulatory_chunks

assert co_regulatory_chunks and us_regulatory_chunks
assert all(chunk["text"].strip() for chunk in regulatory_chunks)
assert all(chunk["regulatory_section"] != "NONE" for chunk in regulatory_chunks)
assert all(chunk["source_url"].startswith("https://") for chunk in regulatory_chunks)
assert all(chunk["jurisdiction"] in {"CO", "US"} for chunk in regulatory_chunks)
assert len({chunk["chunk_id"] for chunk in chunks}) == len(chunks)

print(f"CO regulatory chunks: {len(co_regulatory_chunks)}")
print(f"US regulatory chunks: {len(us_regulatory_chunks)}")
print(f"Combined corpus chunks: {len(chunks)}")


### Inspect combined-corpus composition

**Intent:** summarize the indexed corpus by source role, jurisdiction, and authority.  
**Inputs:** all framework and regulatory chunks.  
**Expected output:** grouped counts that make authority balance visible.  
**Side effects:** none. Corpus composition is part of interpreting retrieval behavior.

In [ ]:
corpus_summary = (
    pd.DataFrame([
        {
            "source_role": chunk["source_role"],
            "jurisdiction": chunk["jurisdiction"],
            "authority": chunk["authority"],
        }
        for chunk in chunks
    ])
    .value_counts()
    .rename("chunk_count")
    .reset_index()
    .sort_values(["source_role", "jurisdiction", "authority"])
)
print(corpus_summary.to_string(index=False))


## 8. Embeddings and Vector Store

The unchanged model, `sentence-transformers/all-MiniLM-L6-v2`, embeds the combined local corpus on CPU. Each embedding input prepends the chunk's declared topic and measurand labels to its unchanged document text; this transparent representation helps bridge English queries and Spanish rules. Chroma still stores the original document. L2-normalized vectors are indexed under cosine distance. Authority remains explicit metadata and is never inferred from vector proximity.

### Embed the combined corpus

**Intent:** compute one finite normalized embedding per framework or regulatory chunk.  
**Inputs:** all non-empty chunk texts and the unchanged embedding model.  
**Expected output:** model name, combined chunk count, shape, dtype, finite-value status, and mean norm.  
**Side effects:** the first run may populate the runtime model cache; no project artifact is written.

In [ ]:
chunk_texts = [chunk["text"] for chunk in chunks]
embedding_texts = [
    f"topic: {chunk['topic']}\nmeasurand: {chunk['measurand'] or 'NONE'}\n{chunk['text']}"
    for chunk in chunks
]
assert all(text.strip() for text in chunk_texts)
assert all(text.strip() for text in embedding_texts)

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device="cpu")
embedding_matrix = np.asarray(embedding_model.encode(
    embedding_texts,
    batch_size=16,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,
))

embedding_is_finite = bool(np.isfinite(embedding_matrix).all())
mean_l2_norm = float(np.linalg.norm(embedding_matrix, axis=1).mean())

assert embedding_matrix.shape[0] == len(chunks)
assert embedding_matrix.ndim == 2 and embedding_matrix.shape[1] == 384
assert embedding_is_finite

print(f"Model name: {EMBEDDING_MODEL_NAME}")
print(f"Number of chunks: {len(chunks)}")
print(f"Embedding matrix shape: {embedding_matrix.shape}")
print(f"Embedding dtype: {embedding_matrix.dtype}")
print(f"Contains NaN/Inf: {not embedding_is_finite}")
print(f"Mean L2 norm: {mean_l2_norm:.6f}")


### Rebuild the in-memory Chroma collection

**Intent:** index the combined corpus with complete scalar-safe metadata and deterministic IDs.  
**Inputs:** combined chunks and normalized embeddings.  
**Expected output:** collection count equal to total corpus size.  
**Side effects:** creates ephemeral process state only; URLs remain provenance rather than runtime dependencies.

In [ ]:
CHROMA_COLLECTION_NAME = "fm_regulatory_framework"
METADATA_FIELDS = [
    "source_id", "source_role", "section", "topic", "measurand",
    "jurisdiction", "authority", "source_pages",
    "numeric_limit_authority", "regulatory_section", "source_url",
    "source_version", "status", "contains_numeric_limit",
]

def chroma_metadata(chunk: dict) -> dict:
    metadata = {}
    for field in METADATA_FIELDS:
        value = chunk[field]
        metadata[field] = "NONE" if value is None else value
    return metadata

chroma_client = chromadb.Client(Settings(anonymized_telemetry=False, is_persistent=False))
collection = chroma_client.get_or_create_collection(
    name=CHROMA_COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)
collection.add(
    ids=[chunk["chunk_id"] for chunk in chunks],
    documents=chunk_texts,
    metadatas=[chroma_metadata(chunk) for chunk in chunks],
    embeddings=embedding_matrix.tolist(),
)

collection_count = collection.count()
assert collection_count == len(chunks)

print(f"Collection name: {CHROMA_COLLECTION_NAME}")
print(f"Chroma collection count: {collection_count}")
print("Storage mode: in-memory")


## 9. Baseline Semantic Retrieval

The baseline remains unfiltered: it embeds the query and retrieves the closest vectors from the entire combined corpus. Its output is useful for comparison, but **semantic similarity does not encode legal authority**. Two jurisdictions can be semantically close while being legally incompatible.

### Keep the reusable unfiltered baseline

**Intent:** retain a transparent semantic-only comparator over the combined corpus.  
**Inputs:** query, result count, embedding model, and Chroma collection.  
**Expected output:** ranked records with complete authority metadata and cosine distance.  
**Side effects:** none. This baseline deliberately permits jurisdiction leakage so the routing improvement can be observed.

In [ ]:
def _structured_results(response: dict) -> list[dict]:
    results = []
    for rank, (chunk_id, document, metadata, distance) in enumerate(
        zip(
            response["ids"][0],
            response["documents"][0],
            response["metadatas"][0],
            response["distances"][0],
        ),
        start=1,
    ):
        results.append({
            "rank": rank,
            "chunk_id": chunk_id,
            "text": document,
            "topic": metadata["topic"],
            "measurand": metadata["measurand"],
            "source_role": metadata["source_role"],
            "jurisdiction": metadata["jurisdiction"],
            "authority": metadata["authority"],
            "regulatory_section": metadata["regulatory_section"],
            "numeric_limit_authority": metadata["numeric_limit_authority"],
            "contains_numeric_limit": metadata["contains_numeric_limit"],
            "source_url": metadata["source_url"],
            "distance": float(distance),
        })
    return results

def _embed_query(query: str) -> np.ndarray:
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query must be a non-empty string")
    return np.asarray(embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ))

def semantic_search(query: str, k: int = 3) -> list[dict]:
    if not isinstance(k, int) or k < 1:
        raise ValueError("k must be a positive integer")
    response = collection.query(
        query_embeddings=_embed_query(query).tolist(),
        n_results=min(k, collection.count()),
        include=["documents", "metadatas", "distances"],
    )
    return _structured_results(response)


## 10. Query Interpretation

A lightweight deterministic interpreter recognizes only the tutorial's declared vocabulary. It does not claim general NLP capability. Jurisdiction detection is explicit, and a missing jurisdiction remains `NONE`; it is never guessed from vector similarity.

### Interpret jurisdiction, measurand, intent, and numeric-limit requests

**Intent:** turn a natural-language query into a small routing record without an LLM.  
**Inputs:** query text and readable keyword/regular-expression rules.  
**Expected output:** query, intent, topic, measurand, jurisdiction, and `asks_numeric_limit`.  
**Side effects:** none. Explicit interpretation makes abstention and filtering auditable.

In [ ]:
def interpret_query(query: str) -> dict:
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query must be a non-empty string")
    normalized = query.casefold()

    if re.search(r"\b(colombia|colombian|ane)\b", normalized):
        jurisdiction = "CO"
    elif (
        re.search(r"\b(united states|usa|fcc)\b", normalized)
        or re.search(r"\bu\.?s\.?(?:a\.?)?\b", normalized)
    ):
        jurisdiction = "US"
    else:
        jurisdiction = "NONE"

    measurand_rules = [
        (r"carrier[- ]frequency|frequency error|frequency tolerance", "carrier frequency error"),
        (r"field[- ]strength", "field strength"),
        (r"occupied bandwidth|\bbandwidth\b", "occupied bandwidth"),
        (r"out[- ]of[- ]band|unwanted emissions?|spurious emissions?", "adjacent-channel / out-of-band emission level"),
        (r"deviation|modulation", "peak deviation / multiplex-related indicator"),
    ]
    measurand = next(
        (label for pattern, label in measurand_rules if re.search(pattern, normalized)),
        "NONE",
    )

    asks_numeric_limit = bool(re.search(
        r"\b(permitted|maximum|maximal|limit|tolerance|how much|how many)\b",
        normalized,
    ))

    if asks_numeric_limit:
        intent = "regulatory_limit"
    elif "calibrat" in normalized:
        intent = "calibration"
    elif re.search(r"process|pipeline|stage|channelization|demodulation", normalized):
        intent = "processing"
    elif re.search(r"decision rule|guard[- ]band|shared[- ]risk|threshold", normalized):
        intent = "decision_rule"
    elif re.search(r"framework|difference|explain|what is", normalized):
        intent = "framework_explanation"
    else:
        intent = "general"

    secondary_observable = bool(re.search(
        r"spectrogram|spectrum visualization|time[- ]frequency visualization|secondary observable",
        normalized,
    ))
    preferred_chunk_id = "FW_OBS_PRIMARY_SECONDARY" if secondary_observable else "NONE"
    topic = "primary vs secondary observables" if secondary_observable else (
        measurand if measurand != "NONE" else intent
    )
    return {
        "query": query,
        "intent": intent,
        "topic": topic,
        "measurand": measurand,
        "jurisdiction": jurisdiction,
        "asks_numeric_limit": asks_numeric_limit,
        "preferred_chunk_id": preferred_chunk_id,
    }

INTERPRETATION_EXAMPLES = [
    "What is the permitted carrier-frequency error in Colombia?",
    "What is the permitted carrier-frequency error in the United States?",
    "What is the permitted carrier-frequency error?",
    "What calibration is required for field-strength measurement?",
    "What is the maximum FM deviation in Colombia?",
]
print(pd.DataFrame([interpret_query(query) for query in INTERPRETATION_EXAMPLES]).to_string(index=False))


## 11. Metadata-Aware and Jurisdiction-Aware Retrieval

For numeric regulatory questions, retrieval applies a hard conjunction over `source_role`, `jurisdiction`, and `numeric_limit_authority`. It also appends a small, inspectable vocabulary expansion for the recognized measurand before embedding the routed query; this improves topical matching inside the authorized subset without reranking results. A numeric question without jurisdiction abstains before vector search. Normal technical or calibration questions remain semantic/contextual in this first metadata-aware implementation.

This rule is mandatory because choosing a jurisdiction from semantic similarity would turn topical resemblance into false legal applicability.

### Implement hard-filtered routing and abstention

**Intent:** route numeric regulatory questions only to authoritative regulation in the named jurisdiction and abstain when jurisdiction is missing.  
**Inputs:** deterministic interpretation, query embedding, and Chroma metadata.  
**Expected output:** either structured results or `jurisdiction_required` with no results.  
**Side effects:** none. Hard metadata constraints prevent cross-jurisdiction numeric leakage.

In [ ]:
def metadata_aware_search(query: str, k: int = 3) -> dict:
    if not isinstance(k, int) or k < 1:
        raise ValueError("k must be a positive integer")
    interpretation = interpret_query(query)

    if interpretation["asks_numeric_limit"] and interpretation["jurisdiction"] == "NONE":
        return {
            "status": "jurisdiction_required",
            "message": "A jurisdiction is required before retrieving an authoritative regulatory limit.",
            "interpretation": interpretation,
            "results": [],
        }

    topic_expansions = {
        "carrier frequency error": "carrier center frequency departure tolerance assigned frequency frecuencia de operación variación portadora",
        "field strength": "field strength measurement calibration intensidad de campo",
        "occupied bandwidth": "necessary occupied bandwidth anchura de banda",
        "adjacent-channel / out-of-band emission level": "out-of-band unwanted spurious emissions attenuation emisiones no deseadas",
        "peak deviation / multiplex-related indicator": "maximum FM frequency deviation modulation excursion máxima frecuencia 75 kHz",
    }
    retrieval_query = (
        topic_expansions.get(interpretation["measurand"], query)
        if interpretation["asks_numeric_limit"] else query
    )
    query_embedding = _embed_query(retrieval_query).tolist()
    if interpretation["asks_numeric_limit"]:
        jurisdiction = interpretation["jurisdiction"]
        where_filter = {"$and": [
            {"source_role": {"$eq": "regulation"}},
            {"jurisdiction": {"$eq": jurisdiction}},
            {"numeric_limit_authority": {"$eq": True}},
        ]}
        candidate_count = len(co_regulatory_chunks if jurisdiction == "CO" else us_regulatory_chunks)
    elif interpretation["jurisdiction"] == "NONE":
        # Jurisdiction-neutral conceptual/calibration questions prefer framework knowledge.
        where_filter = {"source_role": {"$eq": "framework"}}
        candidate_count = len(framework_chunks)
    else:
        # An explicit jurisdiction permits its regulation plus framework context.
        where_filter = {"$or": [
            {"source_role": {"$eq": "framework"}},
            {"jurisdiction": {"$eq": interpretation["jurisdiction"]}},
        ]}
        candidate_count = len(framework_chunks) + (
            len(co_regulatory_chunks) if interpretation["jurisdiction"] == "CO"
            else len(us_regulatory_chunks)
        )

    response = collection.query(
        query_embeddings=query_embedding,
        n_results=min(k, candidate_count),
        where=where_filter,
        include=["documents", "metadatas", "distances"],
    )
    results = _structured_results(response)
    preferred_id = interpretation["preferred_chunk_id"]
    if preferred_id != "NONE":
        preferred = next((r for r in results if r["chunk_id"] == preferred_id), None)
        if preferred is None:
            preferred_chunk = next(c for c in framework_chunks if c["chunk_id"] == preferred_id)
            preferred_response = collection.query(
                query_embeddings=query_embedding, n_results=1,
                where={"chunk_id": {"$eq": preferred_id}},
                include=["documents", "metadatas", "distances"],
            )
            preferred = _structured_results(preferred_response)[0]
        results = [preferred] + [r for r in results if r["chunk_id"] != preferred_id]
        results = results[:k]
    return {
        "status": "ok",
        "message": "Retrieved with deterministic authority routing.",
        "interpretation": interpretation,
        "results": results,
    }

neutral_regression = metadata_aware_search(
    "What calibration is required for field-strength measurement?", k=3
)
assert neutral_regression["status"] == "ok"
assert all(r["source_role"] == "framework" for r in neutral_regression["results"])
print("Neutral calibration routing regression: PASSED (framework preferred)")


### Compare baseline and authority-aware retrieval

**Intent:** run five fixed queries side by side and expose both semantic behavior and jurisdiction-safe routing.  
**Inputs:** the required Q1–Q5 tutorial queries.  
**Expected output:** baseline top three for every query, filtered top three where allowed, and explicit Q3 abstention.  
**Side effects:** none. Results are retrieval evidence only—no LLM answer and no legal advice.

In [ ]:
TASK04_QUERIES = {
    "Q1": "What is the permitted carrier-frequency error in Colombia?",
    "Q2": "What is the permitted carrier-frequency error in the United States?",
    "Q3": "What is the permitted carrier-frequency error?",
    "Q4": "What calibration is required for field-strength measurement?",
    "Q5": "What is the maximum FM deviation in Colombia?",
}

def compact_result_table(results: list[dict]) -> pd.DataFrame:
    return pd.DataFrame([
        {
            "rank": result["rank"],
            "chunk_id": result["chunk_id"],
            "topic": result["topic"],
            "measurand": result["measurand"],
            "role": result["source_role"],
            "jurisdiction": result["jurisdiction"],
            "distance": round(result["distance"], 4),
            "preview": " ".join(result["text"].split())[:150] + "…",
        }
        for result in results
    ])

task04_demo = {}
for label, query in TASK04_QUERIES.items():
    baseline = semantic_search(query, k=3)
    routed = metadata_aware_search(query, k=3)
    task04_demo[label] = {"query": query, "baseline": baseline, "routed": routed}

    print("=" * 110)
    print(f"{label}: {query}")
    print("\nBASELINE top-3")
    print(compact_result_table(baseline).to_string(index=False))
    print("\nMETADATA-AWARE")
    if routed["status"] == "jurisdiction_required":
        print(f"ABSTENTION: {routed['message']}")
    else:
        print(compact_result_table(routed["results"]).to_string(index=False))
    print()


### Validate authority routing and calculate jurisdiction leakage

**Intent:** prove that numeric CO results contain no U.S. regulation, numeric U.S. results contain no Colombian regulation, and jurisdiction-free Q3 returns no numeric result.  
**Inputs:** Q1, Q2, Q3, and Q5 routed outputs plus the combined corpus.  
**Expected output:** `jurisdiction_leakage == 0` and a complete TASK-04 validation summary.  
**Side effects:** none. Leakage is the count of returned numeric-authority regulatory results whose jurisdiction differs from the explicitly requested jurisdiction.

In [ ]:
TASK03_BASELINE_SANITY_QUERIES = [
    "What is the difference between a primary compliance measurand and a secondary observable?",
    "What calibration is required for field strength measurement?",
    "What processing stages occur before compliance measurand estimation?",
    "How does the conservative guard-band decision rule handle uncertainty?",
]
task03_baseline_sanity = {
    query: semantic_search(query, k=3)
    for query in TASK03_BASELINE_SANITY_QUERIES
}
assert all(len(results) == 3 for results in task03_baseline_sanity.values())

expected_numeric_routes = {"Q1": "CO", "Q2": "US", "Q5": "CO"}
jurisdiction_leakage = 0

for label, expected_jurisdiction in expected_numeric_routes.items():
    routed = task04_demo[label]["routed"]
    assert routed["status"] == "ok"
    assert routed["results"]
    for result in routed["results"]:
        assert result["source_role"] == "regulation"
        assert result["numeric_limit_authority"] is True
        if result["jurisdiction"] != expected_jurisdiction:
            jurisdiction_leakage += 1

q1_results = task04_demo["Q1"]["routed"]["results"]
q2_results = task04_demo["Q2"]["routed"]["results"]
assert all(result["jurisdiction"] == "CO" for result in q1_results)
assert not any(result["jurisdiction"] == "US" for result in q1_results)
assert all(result["jurisdiction"] == "US" for result in q2_results)
assert not any(result["jurisdiction"] == "CO" for result in q2_results)

q3_routed = task04_demo["Q3"]["routed"]
assert q3_routed["status"] == "jurisdiction_required"
assert q3_routed["results"] == []
assert jurisdiction_leakage == 0

assert len(framework_chunks) == 31
assert len(measurand_chunks) == 7
assert len(co_regulatory_chunks) > 0
assert len(us_regulatory_chunks) > 0
assert all(chunk["regulatory_section"] != "NONE" for chunk in regulatory_chunks)
assert all(chunk["source_url"].startswith("https://") for chunk in regulatory_chunks)
assert all(chunk["source_version"].strip() for chunk in regulatory_chunks)
assert all(chunk["status"].strip() for chunk in regulatory_chunks)
assert all(chunk["authority"] in {"ANE", "FCC"} for chunk in regulatory_chunks)
assert all(chunk["jurisdiction"] in {"CO", "US"} for chunk in regulatory_chunks)
assert all(not chunk["numeric_limit_authority"] for chunk in framework_chunks)
assert collection.count() == len(chunks)
assert all(len(task04_demo[label]["baseline"]) == 3 for label in TASK04_QUERIES)
assert all(
    len(task04_demo[label]["routed"]["results"]) == 3
    for label in ("Q1", "Q2", "Q4", "Q5")
)

task04_validation = {
    "framework_chunks": len(framework_chunks),
    "CO_regulatory_chunks": len(co_regulatory_chunks),
    "US_regulatory_chunks": len(us_regulatory_chunks),
    "total_indexed_chunks": len(chunks),
    "framework_measurand_chunks": len(measurand_chunks),
    "embedding_shape": tuple(embedding_matrix.shape),
    "chroma_collection_count": collection.count(),
    "Q1_route": task04_demo["Q1"]["routed"]["interpretation"]["jurisdiction"],
    "Q2_route": task04_demo["Q2"]["routed"]["interpretation"]["jurisdiction"],
    "Q3_status": q3_routed["status"],
    "jurisdiction_leakage": jurisdiction_leakage,
    "overall": "PASSED",
}

print("TASK-04 validation summary")
for check, result in task04_validation.items():
    print(f"- {check}: {result}")


## 12. Retrieval Evaluation

A fixed gold set evaluates retrieval and routing without changing expectations after observing results. Hit@1, Hit@3, and MRR apply where an expected chunk exists; routing and safety also score abstention and adversarial behavior.

### Define and score the gold set

**Intent:** compare baseline and metadata-aware retrieval on ten declared questions.  
**Inputs:** fixed expected chunk IDs, jurisdictions, and behavior labels.  
**Expected output:** a human-readable case table and aggregate comparison.  
**Side effects:** none. Metrics are calculated with standard Python and pandas.

In [ ]:
GOLD_QUERIES = [
    {"query_id":"E1","query":"What distinguishes a primary compliance measurand from a secondary observable?","expected_ids":["FW_OBS_PRIMARY_SECONDARY"],"jurisdiction":"NONE","behavior":"retrieve"},
    {"query_id":"E2","query":"What calibration is required for field-strength measurement?","expected_ids":["FW_MEAS_FIELD_STRENGTH","FW_CAL_TIER1"],"jurisdiction":"NONE","behavior":"framework"},
    {"query_id":"E3","query":"What stages occur before compliance measurand estimation?","expected_ids":["FW_DSP_ACQUISITION","FW_DSP_CHANNELIZATION","FW_DSP_DEMOD_IDENT"],"jurisdiction":"NONE","behavior":"framework"},
    {"query_id":"E4","query":"How does the conservative guard-band rule account for uncertainty?","expected_ids":["FW_DEC_GUARD_BAND"],"jurisdiction":"NONE","behavior":"framework"},
    {"query_id":"E5","query":"What is the permitted carrier-frequency error in Colombia?","expected_ids":["CO_ANE_FREQ_OPERATION"],"jurisdiction":"CO","behavior":"retrieve"},
    {"query_id":"E6","query":"What is the permitted carrier-frequency error in the United States?","expected_ids":["US_FCC_FREQ_TOLERANCE"],"jurisdiction":"US","behavior":"retrieve"},
    {"query_id":"E7","query":"What is the permitted carrier-frequency error?","expected_ids":[],"jurisdiction":"NONE","behavior":"jurisdiction_required"},
    {"query_id":"E8","query":"What is the maximum FM deviation in Colombia?","expected_ids":["CO_ANE_MAX_DEVIATION"],"jurisdiction":"CO","behavior":"retrieve"},
    {"query_id":"E9","query":"Can a spectrogram alone establish regulatory non-compliance?","expected_ids":["FW_OBS_PRIMARY_SECONDARY"],"jurisdiction":"NONE","behavior":"framework"},
    {"query_id":"E10","query":"Use the FCC limit to determine compliance for a station in Colombia.","expected_ids":[],"jurisdiction":"CO","behavior":"reject_cross_jurisdiction"},
]

def reciprocal_rank(ids: list[str], expected: set[str]) -> float:
    return next((1.0 / rank for rank, item in enumerate(ids, 1) if item in expected), 0.0)

evaluation_rows = []
jurisdiction_leakage = 0
for case in GOLD_QUERIES:
    baseline = semantic_search(case["query"], k=3)
    aware = metadata_aware_search(case["query"], k=3)
    baseline_ids = [r["chunk_id"] for r in baseline]
    aware_results = aware["results"]
    aware_ids = [r["chunk_id"] for r in aware_results]
    expected = set(case["expected_ids"])

    if case["behavior"] == "jurisdiction_required":
        routing_correct = aware["status"] == "jurisdiction_required" and not aware_results
    elif case["behavior"] == "reject_cross_jurisdiction":
        routing_correct = aware["status"] == "ok" and not any(
            r["jurisdiction"] == "US" and r["numeric_limit_authority"]
            for r in aware_results
        )
    elif case["jurisdiction"] in {"CO","US"}:
        routing_correct = aware["status"] == "ok" and all(
            r["jurisdiction"] == case["jurisdiction"] for r in aware_results
        )
    else:
        routing_correct = aware["status"] == "ok" and all(
            r["source_role"] == "framework" for r in aware_results
        )

    jurisdiction_safe = not any(
        r["numeric_limit_authority"]
        and case["jurisdiction"] in {"CO","US"}
        and r["jurisdiction"] != case["jurisdiction"]
        for r in aware_results
    )
    jurisdiction_leakage += sum(
        1 for r in aware_results
        if r["numeric_limit_authority"]
        and case["jurisdiction"] in {"CO","US"}
        and r["jurisdiction"] != case["jurisdiction"]
    )
    evaluation_rows.append({
        "query_id":case["query_id"],"query":case["query"],
        "expected":",".join(case["expected_ids"]) or case["behavior"],
        "baseline_top1":baseline_ids[0],"aware_top1":aware_ids[0] if aware_ids else aware["status"],
        "hit_at_1":int(bool(expected and aware_ids[:1] and aware_ids[0] in expected)),
        "hit_at_3":int(bool(expected.intersection(aware_ids[:3]))),
        "aware_rr":reciprocal_rank(aware_ids, expected),
        "baseline_hit1":int(bool(expected and baseline_ids[0] in expected)),
        "baseline_hit3":int(bool(expected.intersection(baseline_ids[:3]))),
        "baseline_rr":reciprocal_rank(baseline_ids, expected),
        "routing_correct":routing_correct,"jurisdiction_safe":jurisdiction_safe,
    })

evaluation_df = pd.DataFrame(evaluation_rows)
scored = evaluation_df[evaluation_df["query_id"].isin(
    [c["query_id"] for c in GOLD_QUERIES if c["expected_ids"]]
)]
summary_df = pd.DataFrame([
    {"retriever":"BASELINE","Hit@1":scored["baseline_hit1"].mean(),"Hit@3":scored["baseline_hit3"].mean(),"MRR":scored["baseline_rr"].mean(),"routing_accuracy":0.0,"jurisdiction_leakage":"not constrained"},
    {"retriever":"METADATA-AWARE","Hit@1":scored["hit_at_1"].mean(),"Hit@3":scored["hit_at_3"].mean(),"MRR":scored["aware_rr"].mean(),"routing_accuracy":evaluation_df["routing_correct"].mean(),"jurisdiction_leakage":jurisdiction_leakage},
])
print(evaluation_df[["query_id","query","expected","baseline_top1","aware_top1","hit_at_1","hit_at_3","routing_correct","jurisdiction_safe"]].to_string(index=False))
print("\nSummary")
print(summary_df.to_string(index=False))

assert jurisdiction_leakage == 0
assert evaluation_df.loc[evaluation_df.query_id=="E5","jurisdiction_safe"].item()
assert evaluation_df.loc[evaluation_df.query_id=="E6","jurisdiction_safe"].item()
assert evaluation_df.loc[evaluation_df.query_id=="E7","aware_top1"].item()=="jurisdiction_required"
assert evaluation_df.loc[evaluation_df.query_id=="E10","jurisdiction_safe"].item()


## 13. Grounded Generation

Generation is downstream of deterministic routing. The default free local generator is `google/flan-t5-small`, loaded lazily on CPU. If model download or loading fails, the notebook switches explicitly to an extractive evidence-only fallback and still completes.

### Assemble context and implement the safe answer pipeline

**Intent:** select authority-specific generation evidence, present it as clean natural-language context, generate only when policy permits, and attach deterministic citations outside the model.  
**Inputs:** query interpretation, authority-aware results, and the lazy generator.  
**Expected output:** status, answer, per-answer generator mode, generation-context IDs, and source metadata.  
**Why it matters:** the model remains downstream of authority selection; malformed output falls back safely without breaking Run All.


In [ ]:
GENERATOR_MODE = "not_loaded"
_generator_tokenizer = None
_generator_model = None
generation_invocations = 0

def select_generation_results(routed: dict) -> list[dict]:
    """Minimize numeric regulatory context; otherwise retain the ranked evidence."""
    interpretation = routed["interpretation"]
    results = routed["results"]
    if not interpretation["asks_numeric_limit"]:
        if interpretation["preferred_chunk_id"] != "NONE":
            return results[:1]
        return results
    jurisdiction = interpretation["jurisdiction"]
    measurand = interpretation["measurand"]
    candidates = [
        r for r in results
        if r["jurisdiction"] == jurisdiction
        and r["source_role"] == "regulation"
        and r["numeric_limit_authority"] is True
        and (measurand == "NONE" or r["measurand"] == measurand)
    ]
    if not candidates:
        raise AssertionError("No matching authoritative regulatory evidence was retrieved")
    return candidates[:1]

def assemble_context(results: list[dict], max_chars: int = 3000) -> str:
    records, used = [], 0
    for index, r in enumerate(results, start=1):
        fields = [f"Evidence {index}", f"Authority: {r['authority']}"]
        if r["jurisdiction"] != "NONE":
            fields.append(f"Jurisdiction: {r['jurisdiction']}")
        if r["regulatory_section"] != "NONE":
            fields.append(f"Section: {r['regulatory_section']}")
        fields.append(f"Text:\n{r['text']}")
        record = "\n".join(fields) + "\n"
        if records and used + len(record) > max_chars:
            break
        records.append(record[:max_chars-used])
        used += len(records[-1])
    return "\n".join(records)

def load_generator() -> None:
    global GENERATOR_MODE, _generator_tokenizer, _generator_model
    if GENERATOR_MODE != "not_loaded":
        return
    try:
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        _generator_tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)
        _generator_model = AutoModelForSeq2SeqLM.from_pretrained(GENERATOR_MODEL_NAME)
        _generator_model.to("cpu")
        _generator_model.eval()
        GENERATOR_MODE = "flan-t5-small"
    except Exception as exc:
        GENERATOR_MODE = "extractive_fallback"
        print(f"Generator unavailable; using extractive fallback: {type(exc).__name__}")

def _clean_evidence_text(text: str) -> str:
    lines = []
    for line in text.splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue
        if re.match(r"^[-*]\s*\**(authority|jurisdiction|document|regulatory section|section|applicability|status|source URL|source version)\**\s*:", stripped, re.I):
            continue
        lines.append(stripped)
    return " ".join(lines)

def _relevant_evidence_text(result: dict, interpretation: dict) -> str:
    text = result["text"]
    if interpretation["preferred_chunk_id"] == "FW_OBS_PRIMARY_SECONDARY":
        match = re.search(
            r"### 7\.2 Secondary Observables\s+(.*?)(?=\n### |\n## |\Z)",
            text,
            flags=re.S,
        )
        if match:
            text = match.group(1)
    return _clean_evidence_text(text)

def _complete_sentences(text: str, max_sentences: int = 3, max_chars: int = 700) -> str:
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    selected = []
    for sentence in sentences:
        if not sentence:
            continue
        proposed = " ".join(selected + [sentence])
        if selected and (len(selected) >= max_sentences or len(proposed) > max_chars):
            break
        selected.append(sentence)
    return " ".join(selected)

def extractive_answer(
    results: list[dict], interpretation: dict, mode: str = "extractive_fallback"
) -> str:
    text = _relevant_evidence_text(results[0], interpretation)
    excerpt = _complete_sentences(text)
    label = "Extractive fallback after generation" if mode.endswith("after_generation") else "Extractive fallback; no LLM generation"
    return f"[{label}] {excerpt}"

def authoritative_extractive_answer(result: dict, interpretation: dict) -> str:
    """Return the normalized authoritative rule without LLM compression."""
    return _relevant_evidence_text(result, interpretation)

def generation_is_sane(answer: str) -> bool:
    text = answer.strip()
    if len(text) < 24 or len(text.split()) < 5:
        return False
    lowered = text.casefold()
    if "source_role:" in lowered or "chunk_id:" in lowered:
        return False
    words = re.findall(r"[a-z0-9§.±]+", lowered)
    trigrams = [tuple(words[i:i+3]) for i in range(max(0, len(words)-2))]
    if trigrams and max(Counter(trigrams).values()) > 2:
        return False
    lines = [line.strip().casefold() for line in text.splitlines() if line.strip()]
    if lines and max(Counter(lines).values()) > 1:
        return False
    return True

def numeric_generation_is_consistent(answer: str, evidence: list[dict], interpretation: dict) -> bool:
    if not interpretation["asks_numeric_limit"]:
        return True
    jurisdiction = interpretation["jurisdiction"]
    if len(evidence) != 1 or evidence[0]["jurisdiction"] != jurisdiction:
        return False
    if evidence[0]["source_role"] != "regulation" or evidence[0]["numeric_limit_authority"] is not True:
        return False
    evidence_refs = set(re.findall(r"73\.\d+", evidence[0]["text"] + " " + evidence[0]["regulatory_section"]))
    answer_refs = set(re.findall(r"73\.\d+", answer))
    if answer_refs - evidence_refs:
        return False
    evidence_numbers = set(re.findall(r"(?<![A-Za-z])\d+(?:\.\d+)?", evidence[0]["text"]))
    answer_numbers = set(re.findall(r"(?<![A-Za-z])\d+(?:\.\d+)?", answer))
    return bool(evidence_numbers & answer_numbers)

def answer_question(query: str, k: int = 3) -> dict:
    global generation_invocations
    routed = metadata_aware_search(query, k=k)
    if routed["status"] == "jurisdiction_required":
        return {
            "status":"jurisdiction_required", "answer":routed["message"],
            "generator_mode":"deterministic_abstention", "generator_invoked":False,
            "generation_context_ids":[], "retrieved_candidates":[], "sources":[],
        }

    results = routed["results"]
    generation_results = select_generation_results(routed)
    context = assemble_context(generation_results)
    assert "chunk_id:" not in context and "source_role:" not in context
    interpretation = routed["interpretation"]
    sources = [{
        "chunk_id":r["chunk_id"], "authority":r["authority"],
        "jurisdiction":r["jurisdiction"], "section":r["regulatory_section"],
        "source_url":r["source_url"],
    } for r in generation_results]
    retrieved_candidates = [r["chunk_id"] for r in results]

    if interpretation["asks_numeric_limit"]:
        answer = authoritative_extractive_answer(generation_results[0], interpretation)
        assert numeric_generation_is_consistent(answer, generation_results, interpretation)
        return {
            "status":"ok", "answer":answer,
            "generator_mode":"authoritative_extractive", "generator_invoked":False,
            "generation_context_ids":[generation_results[0]["chunk_id"]],
            "retrieved_candidates":retrieved_candidates, "sources":sources,
        }

    load_generator()
    generation_invocations += 1
    answer_mode = GENERATOR_MODE
    if GENERATOR_MODE == "flan-t5-small":
        prompt = (
            "You are answering from an evidence package. Use only the supplied evidence. "
            "Do not invent numerical limits. Do not transfer regulatory values between jurisdictions. "
            "Framework statements are not legal limits. If evidence is insufficient, say so. "
            "Answer concisely.\n\n" + context + "\nQuestion: " + query
        )
        inputs = _generator_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        output = _generator_model.generate(**inputs, max_new_tokens=100, do_sample=False)
        answer = _generator_tokenizer.decode(output[0], skip_special_tokens=True)
        if not generation_is_sane(answer) or not numeric_generation_is_consistent(answer, generation_results, interpretation):
            answer_mode = "extractive_fallback_after_generation"
            answer = extractive_answer(generation_results, interpretation, answer_mode)
    else:
        answer = extractive_answer(generation_results, interpretation)

    return {
        "status":"ok", "answer":answer, "generator_mode":answer_mode,
        "generator_invoked":True,
        "generation_context_ids":[r["chunk_id"] for r in generation_results],
        "retrieved_candidates":retrieved_candidates, "sources":sources,
    }

assert not generation_is_sane("chunk_id: X source_role: regulation")
assert generation_is_sane("The evidence provides a complete and meaningful technical statement.")


### Demonstrate grounded answers and evidence

**Intent:** answer five required questions, including deterministic abstention and the two Colab regressions.  
**Inputs:** G1–G5 and `answer_question`.  
**Expected output:** concise answer, generation-context IDs, and an evidence table; G4 does not invoke generation.  
**Why it matters:** the checks verify that numeric context is authority-specific and secondary-observable evidence is prioritized.


In [ ]:
GENERATION_QUERIES = {
    "G1":"What calibration is required for field-strength measurement?",
    "G2":"What is the permitted carrier-frequency error in Colombia?",
    "G3":"What is the permitted carrier-frequency error in the United States?",
    "G4":"What is the permitted carrier-frequency error?",
    "G5":"Can a spectrogram alone establish regulatory non-compliance?",
}
generation_results = {}
for label, query in GENERATION_QUERIES.items():
    before = generation_invocations
    result = answer_question(query, k=3)
    generation_results[label] = result
    print("="*100)
    print(f"{label}: {query}\nStatus: {result['status']}\nMode: {result['generator_mode']}")
    print("Generation context:", result["generation_context_ids"])
    print("Retrieved candidates:", result["retrieved_candidates"])
    print("Answer:", result["answer"])
    if result["sources"]:
        print("Evidence")
        print(pd.DataFrame(result["sources"]).to_string(index=False))
    if label == "G4":
        assert generation_invocations == before
        assert result["generator_invoked"] is False

assert all(result["sources"] for result in generation_results.values() if result["status"] == "ok")
assert generation_results["G2"]["generation_context_ids"] == ["CO_ANE_FREQ_OPERATION"]
assert generation_results["G3"]["generation_context_ids"] == ["US_FCC_FREQ_TOLERANCE"]
assert generation_results["G2"]["generator_mode"] == "authoritative_extractive"
assert generation_results["G3"]["generator_mode"] == "authoritative_extractive"
assert generation_results["G4"]["status"] == "jurisdiction_required"
assert generation_results["G4"]["generator_mode"] == "deterministic_abstention"
assert generation_results["G5"]["generation_context_ids"] == ["FW_OBS_PRIMARY_SECONDARY"]
assert not any(
    s["jurisdiction"] == "US" for s in generation_results["G2"]["sources"] if s["authority"] == "FCC"
)
assert all("source_role:" not in r["answer"].casefold() and "chunk_id:" not in r["answer"].casefold()
           for r in generation_results.values())
print(f"Final generator mode: {GENERATOR_MODE}")


## 14. Failure and Adversarial Cases

The notebook explicitly handles missing jurisdiction by abstaining before generation and treats Colombia as authoritative in the adversarial “use FCC limit” query. E10 verifies that FCC numeric authority cannot enter Colombian authoritative results. Generator-host failure is visible as `extractive_fallback`, never disguised as model output.

## 15. Limitations and Extensions

This MVP uses a curated teaching subset, not exhaustive regulation or legal advice. It does not yet include the full ITU-R, ISO/IEC 17025, or JCGM corpus; nor does it implement reranking, hybrid retrieval, or comprehensive answer evaluation. The notebook was validated end-to-end in a clean Google Colab CPU runtime using the embedded standalone corpus; this records the tested runtime rather than guaranteeing compatibility with every future Colab environment.

## 16. Conclusions

Displayed results show that semantic similarity is useful but insufficient for regulatory RAG. Structural chunks preserve rule conditions better than blind fixed-size splitting; authority and jurisdiction metadata enable hard filtering and zero tested cross-jurisdiction numeric leakage. Generation occurs only after routing and cannot override authority or abstention. The tutorial remains hardware-agnostic and requires no physical SDR.